In [ ]:
import panel as pn
import pandas as pd
from config.database import SessionLocal
from models import Paciente, Medicamento, PacienteMedicamento
from datetime import date

pn.extension('tabulator', design='material', sizing_mode="stretch_width")

custom_css = """
.premium-btn {
    border-radius: 6px !important;
    font-weight: 600 !important;
    letter-spacing: 0.5px !important;
    transition: all 0.2s ease-in-out !important;
}
.premium-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 4px 12px rgba(0,0,0,0.15) !important;
}
.premium-card-title {
    font-size: 1.1rem !important;
    color: #1e293b !important;
    font-weight: 700 !important;
}
"""
pn.config.raw_css.append(custom_css)

status_alert = pn.pane.Alert("", alert_type="light", visible=False, sizing_mode="stretch_width")

txt_cpf = pn.widgets.TextInput(name="CPF Paciente", placeholder="Ex: 111.111.111-11")
txt_nome = pn.widgets.TextInput(name="Nome Completo", placeholder="Ex: João da Silva")
txt_telefone = pn.widgets.TextInput(name="Telefone", placeholder="(00) 00000-0000")
txt_cidade = pn.widgets.TextInput(name="Cidade", placeholder="Ex: São Paulo")

btn_salvar = pn.widgets.Button(name="Salvar / Atualizar Paciente", button_type="primary", css_classes=['premium-btn'])
btn_excluir = pn.widgets.Button(name="Excluir Paciente", button_type="danger", css_classes=['premium-btn'])
btn_limpar = pn.widgets.Button(name="Limpar", button_type="light", css_classes=['premium-btn'])

tabela_dados = pn.widgets.Tabulator(
    pd.DataFrame(),
    height=250,
    layout='fit_columns',
    selectable='checkbox',
    theme='fast'
)

aba_prontuario_titulo = pn.pane.Markdown(
    "### 🏥 Prontuário Médico: Selecione um paciente primeiro",
    styles={'color': '#64748b'}
)

txt_med_generico = pn.widgets.TextInput(name="Nome Genérico", placeholder="Ex: Paracetamol")
txt_med_comercial = pn.widgets.TextInput(name="Nome Comercial", placeholder="Ex: Tylenol")
date_inicio = pn.widgets.DatePicker(name="Data de Início", value=date.today())
date_fim = pn.widgets.DatePicker(name="Data de Fim (Opcional)", value=None)
txt_frequencia = pn.widgets.TextInput(name="Frequência", placeholder="Ex: 8 em 8 horas")
txt_dosagem = pn.widgets.TextInput(name="Dosagem", placeholder="Ex: 500mg")

btn_salvar_med = pn.widgets.Button(name="Prescrever Medicamento", button_type="success", css_classes=['premium-btn'])
btn_excluir_med = pn.widgets.Button(name="Remover Prescrição", button_type="danger", css_classes=['premium-btn'])

tabela_medicamentos = pn.widgets.Tabulator(
    pd.DataFrame(),
    height=200,
    layout='fit_columns',
    selectable='checkbox',
    theme='fast'
)

def carregar_dados_pacientes():
    db = SessionLocal()
    try:
        query = db.query(Paciente).all()
        dados = [
            {
                "CPF": p.cpf_paciente,
                "Nome": p.nome_completo,
                "Telefone": p.telefone,
                "Cidade": p.cidade
            }
            for p in query
        ]
        tabela_dados.value = pd.DataFrame(dados)
    finally:
        db.close()

def carregar_prontuario(cpf):
    db = SessionLocal()
    try:
        query = db.query(
            PacienteMedicamento,
            Medicamento
        ).join(
            Medicamento,
            PacienteMedicamento.id_medicamento == Medicamento.id_medicamento
        ).filter(
            PacienteMedicamento.cpf_paciente == cpf
        ).all()

        dados = [
            {
                "ID Med": med.id_medicamento,
                "Genérico": med.nome_generico,
                "Comercial": med.nome_comercial,
                "Início": pm.data_inicio,
                "Fim": pm.data_fim,
                "Frequência": pm.frequencia,
                "Dosagem": pm.dosagem
            }
            for pm, med in query
        ]
        tabela_medicamentos.value = pd.DataFrame(dados)
    finally:
        db.close()

@pn.depends(tabela_dados.param.selection, watch=True)
def ao_selecionar_paciente(selection):
    if not selection:
        txt_cpf.value = ""
        txt_nome.value = ""
        txt_telefone.value = ""
        txt_cidade.value = ""
        aba_prontuario_titulo.object = "### 🏥 Prontuário Médico: Selecione um paciente primeiro"
        tabela_medicamentos.value = pd.DataFrame()
        return

    linha = tabela_dados.value.iloc[selection[0]]

    txt_cpf.value = str(linha["CPF"])
    txt_nome.value = str(linha["Nome"])
    txt_telefone.value = str(linha["Telefone"]) if pd.notna(linha["Telefone"]) else ""
    txt_cidade.value = str(linha["Cidade"]) if pd.notna(linha["Cidade"]) else ""

    aba_prontuario_titulo.object = f"### 🏥 Prontuário Médico de: **{linha['Nome']}**"
    carregar_prontuario(str(linha["CPF"]))

def salvar_paciente(event):
    if not txt_cpf.value or not txt_nome.value:
        status_alert.object = "Erro: CPF e Nome são obrigatórios."
        status_alert.alert_type = "danger"
        status_alert.visible = True
        return

    db = SessionLocal()

    try:
        paciente = db.query(Paciente).filter_by(cpf_paciente=txt_cpf.value).first()

        if not paciente:
            paciente = Paciente(cpf_paciente=txt_cpf.value)
            db.add(paciente)
            msg = "Paciente cadastrado com sucesso!"
        else:
            msg = "Paciente atualizado com sucesso!"

        paciente.nome_completo = txt_nome.value
        paciente.telefone = txt_telefone.value
        paciente.cidade = txt_cidade.value

        db.commit()

        status_alert.object = msg
        status_alert.alert_type = "success"
        status_alert.visible = True

        carregar_dados_pacientes()

    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True

    finally:
        db.close()

def prescrever_medicamento(event):
    if not txt_cpf.value:
        status_alert.object = "Selecione um paciente na aba 'Dados Gerais' primeiro."
        status_alert.alert_type = "warning"
        status_alert.visible = True
        return

    db = SessionLocal()

    try:
        med = db.query(Medicamento).filter_by(
            nome_generico=txt_med_generico.value
        ).first()

        if not med:
            med = Medicamento(
                nome_generico=txt_med_generico.value,
                nome_comercial=txt_med_comercial.value
            )
            db.add(med)
            db.flush()

        prescricao = PacienteMedicamento(
            cpf_paciente=txt_cpf.value,
            id_medicamento=med.id_medicamento,
            data_inicio=date_inicio.value,
            data_fim=date_fim.value,
            frequencia=txt_frequencia.value,
            dosagem=txt_dosagem.value
        )

        db.add(prescricao)
        db.commit()

        status_alert.object = "Medicamento prescrito com sucesso!"
        status_alert.alert_type = "success"
        status_alert.visible = True

        carregar_prontuario(txt_cpf.value)

    except Exception as e:
        db.rollback()
        status_alert.object = f"Erro na prescrição: {str(e)}"
        status_alert.alert_type = "danger"
        status_alert.visible = True

    finally:
        db.close()

btn_salvar.on_click(salvar_paciente)
btn_salvar_med.on_click(prescrever_medicamento)

aba_pacientes = pn.Column(
    pn.Row(txt_cpf, txt_nome),
    pn.Row(txt_telefone, txt_cidade),
    pn.Row(btn_salvar, btn_excluir, btn_limpar),
    pn.layout.Divider(),
    "**Pacientes Cadastrados**",
    tabela_dados
)

aba_medicamentos = pn.Column(
    aba_prontuario_titulo,
    pn.Row(txt_med_generico, txt_med_comercial),
    pn.Row(date_inicio, date_fim, txt_frequencia, txt_dosagem),
    pn.Row(btn_salvar_med, btn_excluir_med),
    pn.layout.Divider(),
    "**Medicamentos Ativos**",
    tabela_medicamentos
)

tabs_principal = pn.Tabs(
    ("👤 Dados Gerais", aba_pacientes),
    ("💊 Prontuário", aba_medicamentos),
    dynamic=True
)

formulario_card = pn.Card(
    tabs_principal,
    title="Gestão de Prontuários",
    header_css_classes=["premium-card-title"]
)

btn_pacientes = pn.widgets.Button(
    name="Módulo Pacientes",
    button_type="primary",
    sizing_mode="stretch_width",
    disabled=True
)

btn_cuidadores = pn.widgets.Button(
    name="Módulo Cuidadores",
    button_type="light",
    sizing_mode="stretch_width"
)
btn_cuidadores.js_on_click(code="window.location.href='/tela_cuidador'")

btn_transacoes = pn.widgets.Button(
    name="Módulo Transações",
    button_type="light",
    sizing_mode="stretch_width"
)
btn_transacoes.js_on_click(code="window.location.href='/tela_transacoes'")

sidebar_content = pn.Column(
    pn.pane.Markdown("### 🏥 Menu Principal", margin=(0, 0, 10, 0)),
    btn_pacientes,
    btn_cuidadores,
    btn_transacoes,
    pn.layout.Divider(),
    pn.pane.Markdown(
        "**Usuário:** Admin\n\n**Status:** Online 🟢",
        styles={"color": "#cbd5e1", "font-size": "13px"}
    ),
    sizing_mode="stretch_width"
)

template = pn.template.FastListTemplate(
    title="CareLink | Hub de Saúde",
    header_background="#0f172a",
    accent_base_color="#3b82f6",
    sidebar=[sidebar_content],
    main=[
        status_alert,
        formulario_card
    ]
)

carregar_dados_pacientes()
template.servable()